In [5]:
import numpy as np
import pandas as pd

from src.dataset.dataset_class import GreyhoundRaceDataset


In [2]:
past_dog_infos = pd.read_csv("../data/processed/05_dog_infos_engineered.csv")
race_headers = pd.read_csv("../data/processed/09_graded_a_race_header.csv")

header_source_files = set(race_headers["source_file"].dropna())
race_stats = pd.concat(
    [
        chunk.loc[chunk["source_file"].isin(header_source_files)]
        for chunk in pd.read_csv("../data/processed/10_all_race_infos.csv", chunksize=250_000)
    ],
    ignore_index=True,
)


In [3]:
race_stats_counts = race_stats.groupby("source_file").size()
header_counts = race_headers["source_file"].map(race_stats_counts)

validation_summary = pd.Series({
    "loaded_race_stats_races": int(race_stats_counts.shape[0]),
    "loaded_race_stats_races_with_non_6_dogs": int((race_stats_counts != 6).sum()),
    "graded_header_races": int(len(race_headers)),
    "graded_header_races_with_stats": int(header_counts.notna().sum()),
    "graded_header_missing_stats": int(header_counts.isna().sum()),
    "graded_header_races_with_non_6_dogs": int((header_counts.notna() & header_counts.ne(6)).sum()),
    "graded_header_races_with_exactly_6_dogs": int((header_counts == 6).sum()),
})

invalid_race_stats_examples = race_stats_counts[race_stats_counts != 6].head(10)
validation_summary


loaded_race_stats_races                    382150
loaded_race_stats_races_with_non_6_dogs        46
graded_header_races                        382714
graded_header_races_with_stats             382150
graded_header_missing_stats                   564
graded_header_races_with_non_6_dogs            46
graded_header_races_with_exactly_6_dogs    382104
dtype: int64

In [6]:
train_dataset = GreyhoundRaceDataset(
    race_headers=race_headers,
    race_stats=race_stats,
    past_dog_infos=past_dog_infos,
    seq_len=10,
    strict_six_dogs=False,
    drop_incomplete_races=True,
)

sample = train_dataset[0]
{
    "dataset_len": len(train_dataset),
    "race_stats_shape": sample["race_stats"].shape,
    "dog_history_matrix_shape": sample["dog_history_matrix"].shape,
    "dog_history_lengths": sample["dog_history_lengths"].tolist(),
}


{'dataset_len': 382104,
 'race_stats_shape': (6, 152),
 'dog_history_matrix_shape': (6, 10, 48),
 'dog_history_lengths': [10, 10, 10, 10, 10, 10]}